# Exact Sports Scheduling Project
**Based on Durán et al. (2007)**

### 1. Environment Setup
Run this cell **once** every time you open this notebook. It clones the GitHub repository into the Colab cloud environment, sets up the directory paths, and installs the necessary Operations Research solvers and data libraries.

In [ ]:
import sys
import os

# 1. Clone the repository into the Colab cloud environment
!git clone https://github.com/Anant1607/Sports_Scheduling_Project.git

# 2. Set the working directory to the cloned repository
os.chdir('/content/Sports_Scheduling_Project')

# 3. Add the src folder to the Python path so local modules can be imported
sys.path.append('/content/Sports_Scheduling_Project')

# 4. Install the operations research and data libraries
!pip install "pulp<4.0" highspy pandas pytest

print("Environment setup complete.")

### 2. Team Collaboration Sync
Run this cell whenever a team member pushes new code to the `main` branch on GitHub. It forcefully pulls the latest `.py` files and uses `importlib` to refresh the Python environment so the new logic is applied immediately without needing to restart the entire Colab runtime.

In [ ]:
import importlib
# import model # Uncomment once you create src/model.py
# import solver # Uncomment once you create src/solver.py

# Forcefully pull the latest code from GitHub
!git fetch --all
!git reset --hard origin/main

# Manually reload the modules to apply the latest changes
# importlib.reload(model)
# importlib.reload(solver)

print("Latest code pulled from GitHub and modules reloaded.")

### 3. Pipeline Execution
This cell drives the project. It loads the CSV data, builds the PuLP model, executes the HiGHS solver, and extracts the final schedule.

**CRITICAL:** Before saving a copy of this notebook back to GitHub, always go to **Edit > Clear all outputs** in the top menu to prevent JSON merge conflicts!

In [ ]:
import pandas as pd
import os
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.path.abspath('src'))
import model
import solver
import extractor

# 1. Load the data
data_path = 'data/teams.csv'
try:
    df_teams = pd.read_csv(data_path)
    print(f"Data loaded successfully from {data_path}:\n")
    display(df_teams.head())
except FileNotFoundError:
    print(f"Error: {data_path} not found. Ensure it is pushed to GitHub in the data/ folder.")

# 2. Build the mathematical model
print("\n--- Building Model ---")
pulp_model, decision_vars = model.build_model(df_teams)

# 3. Execute the exact solver
print("\n--- Running Solver ---")
status, solve_time, obj_val = solver.run_exact_solver(pulp_model)

# 4. Extract and save the schedule if Optimal
print("\n--- Extracting Results ---")
if status == 'Optimal':
    # Parse the binary variables into a pandas table
    schedule_df = extractor.extract_schedule(decision_vars, dummy_team=None)

    # 2. Map the Team Names from your CSV
    # NOTE: Change 'TeamName' below to whatever the exact column name is in your teams.csv file
    name_mapping = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

    # Insert the names into the dataframe right next to their IDs
    schedule_df.insert(2, 'Home_Name', schedule_df['Home'].map(name_mapping))
    schedule_df.insert(4, 'Away_Name', schedule_df['Away'].map(name_mapping))

    # Display the final human-readable schedule
    display(schedule_df)

    # Save the file so test_validation.py can read it
    os.makedirs('outputs', exist_ok=True)
    schedule_df.to_csv('outputs/schedule.csv', index=False)
    print("\nSuccess: Schedule saved to outputs/schedule.csv. You can now run the pytest cell.")
else:
    print(f"Extraction skipped. Solver returned status: {status}")

In [ ]:
!pytest tests/test_validation.py -v